Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Conditional Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Instead of the unconditional forecast, ask the model a *what-if* question about the first future step: what does it expect for portfolio NAV and realized volatility **given** something about the other columns at that same step.

The `condition` query mode answers this in closed form from the model's joint distribution at one future position. A request names that position once, by its index into `query_times`, and attaches one condition per column it wants to fix:

- An **equality condition** pins a column to a value (`EqualityCondition`). The pinned column leaves the read-out set, because reading it back would only repeat the request.
- An **interval condition** confines a column to a range whose bounds may be open on either side (`IntervalCondition`). The column stays readable: what comes back is its distribution inside the range.

The cell below asks for the same forecast twice, once without the block and once with it, because a conditional mean is only readable next to the unconditional one: what the what-if is worth is the shift between them, not the level of either.

Every column without a condition is a read-out column, and the response describes the conditional distribution of those columns at the conditioned position only, so `outputs.query_times` has exactly one entry however many `query_times` the request carried.

Every read-out the service serves reads that same conditional, and the sections below take it in turn: as a mean beside the unconditional one, as coherent joint draws, as quantiles inside a bounded band, and as the log density of values you already hold. The last two sections use the plausibility numbers to rank candidate what-ifs against each other.

Whether a deployment can condition depends on the checkpoint's head, so `/healthz` advertises `condition` in `supported_query_modes` and the kinds it answers in `supported_condition_kinds`. The client checks that advertisement before sending, and this notebook reads it explicitly so the check is visible.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import (
    ConditionBlock,
    EqualityCondition,
    JointFMClient,
    plan_forecast_columns,
)

HISTORY_PATH = Path("notebooks/history.csv")
FEATURE_COLUMNS = ["equity_index_level", "treasury_10y_yield", "eur_usd_rate"]
TARGET_COLUMNS = ["portfolio_nav", "realized_volatility"]
PINNED_COLUMN = "equity_index_level"
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
CONDITIONED_STEP = 0
EQUITY_RALLY = 1.02
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))
# Every column a request may read back: an equality condition removes its own
# column from the read-out set, and nothing else does.
READABLE_COLUMNS = [name for name in EXPECTED_COLUMNS if name != PINNED_COLUMN]

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}"
    )
if len(history) != INPUT_STEPS:
    raise ValueError(f"Expected {INPUT_STEPS} history rows, got {len(history)}")

client = JointFMClient.from_env()
health = client.health(cache=True)
if "condition" not in health.supported_query_modes:
    raise RuntimeError(
        f"This deployment serves {list(health.supported_query_modes)} only; "
        "mount a checkpoint whose head can condition"
    )
print("condition kinds served:", list(health.supported_condition_kinds))

plan = plan_forecast_columns(
    health=health,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)

last_equity_level = float(history[PINNED_COLUMN].iloc[-1])
rallied_equity_level = last_equity_level * EQUITY_RALLY
rally = ConditionBlock(
    query_time_index=CONDITIONED_STEP,
    conditions=[EqualityCondition(column=PINNED_COLUMN, value=rallied_equity_level)],
)
baseline = client.forecast_mean(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    seed=7,
)
result = client.forecast_mean(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    seed=7,
    condition=rally,
)
if result.query_times != (QUERY_TIMES[CONDITIONED_STEP],):
    raise ValueError(
        f"Expected the conditioned position alone, got {result.query_times!r}"
    )
if result.plausibility is None:
    raise ValueError("A condition response must carry its plausibility block")
print("log density of the pinned value:", result.plausibility.equality_log_density)
forecast = result.to_pandas_tidy()
expected_forecast_rows = len(plan.requested_columns)
if len(forecast) != expected_forecast_rows:
    raise ValueError(
        f"Expected {expected_forecast_rows} forecast rows, got {len(forecast)}"
    )

# The conditional answers one position, so joining on it keeps exactly the rows
# the two requests have in common.
comparison = baseline.to_pandas_tidy().merge(
    forecast,
    on=["query_time", "requested_column"],
    suffixes=("_unconditional", "_given_the_rally"),
)
comparison["shift"] = (
    comparison["value_given_the_rally"] - comparison["value_unconditional"]
)
comparison

## Reading the plausibility
`result.plausibility.equality_log_density` is the log density the model assigns to the pinned value before conditioning. It separates *the model is confident about NAV given this rally* from *the model finds a rally of this size absurd and is extrapolating*. The service reports the number and never refuses on it; comparing it across candidate pins, or against the density of a pin at the model's own unconditional mean, is the caller's decision.

It is a *density*, not a probability, and `exp()` of it is not one either: an exact value of a continuous column has probability zero, and a density carries the reciprocal units of the column it scores, so quoting the equity index in thousands of points instead of points shifts this number by `log(1000)` and can push its exponential above one. Only differences are unit-free, which is why the sentence above says compare rather than threshold: the gap between two pins on the same column is a log likelihood ratio, and that is what ranks candidate scenarios. When a pin covers several columns at once the number stays a single joint density over all of them — never a per-column value to be multiplied back together, which would assume the columns move independently.

## Scenarios, not summaries
A mean answers *where*, and a band answers *how wide*, but a portfolio question usually needs whole futures: draws. Every row below is one coherent joint scenario, because the service draws the read-out columns together from the same conditional mixture — the NAV and the volatility in one row belong to each other, so a function of several columns can be evaluated row by row. A quantile table cannot answer that: the 90th percentile of NAV and the 90th percentile of volatility need not describe any single future.

`diagnostics.condition_draws` reports how many draws stand behind the answer. Oversized sample requests are split into batches against the deployment's advertised cap and merged locally, and the merged count is what this field reports.

In [ ]:
N_SAMPLES = 8

scenarios = client.forecast_samples(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    n_samples=N_SAMPLES,
    seed=7,
    condition=rally,
)
if scenarios.query_times != (QUERY_TIMES[CONDITIONED_STEP],):
    raise ValueError(
        f"Expected the conditioned position alone, got {scenarios.query_times!r}"
    )
print("draws behind the answer:", scenarios.diagnostics.condition_draws)
scenarios.to_pandas_wide()

## Interval condition
Now confine the 10-year yield to a band around its last observed value instead of pinning it, and pin the equity index at the same time: a request may mix both kinds across the columns of one position. The response then carries `region_log_probability`, and the yield column itself stays readable because its distribution inside the band is a genuine answer.

That number is the log probability of the band **given the pinned equity index**, not the band's own probability, because the two condition kinds compose in a fixed order and the band is measured on the distribution the pin has already reduced. The two numbers therefore chain rather than describe separate things, and adding them gives the plausibility of the whole request — `log(density(pin) * P(band | pin))` — with no independence assumed anywhere. A request carrying only interval conditions has nothing to combine, and its `region_log_probability` alone is the joint probability of everything it asked about.

With one interval column, as here, the region probability is exact — one difference of distribution functions per mixture component — and `diagnostics.interval_estimator` stays empty, because there is no estimate to characterize. Bounding a second column makes the region a box with no closed form, which the service estimates numerically and then reports the accounting for — the next section does exactly that.

A band is also where the quantile read-out earns its place over the mean: what the banded column comes back with is a distribution inside its own range, so the cell asserts every quantile of it lands there.

In [ ]:
from jointfm_client import IntervalCondition

QUANTILES = [0.1, 0.5, 0.9]
YIELD_BAND_HALF_WIDTH = 0.002
# A quantile of the truncated draws can sit on the band edge, and the round trip
# through the response's float encoding may move it by an ulp.
BAND_TOLERANCE = 1e-9

last_yield = float(history["treasury_10y_yield"].iloc[-1])
yield_lower = last_yield - YIELD_BAND_HALF_WIDTH
yield_upper = last_yield + YIELD_BAND_HALF_WIDTH
yield_band = IntervalCondition(
    column="treasury_10y_yield", lower=yield_lower, upper=yield_upper
)
rally_with_yield_band = ConditionBlock(
    query_time_index=CONDITIONED_STEP,
    conditions=[
        EqualityCondition(column=PINNED_COLUMN, value=rallied_equity_level),
        yield_band,
    ],
)
banded = client.forecast_quantiles(
    history,
    query_times=QUERY_TIMES,
    requested_columns=["treasury_10y_yield", *plan.requested_columns],
    columns=plan.columns,
    quantiles=QUANTILES,
    seed=7,
    condition=rally_with_yield_band,
)
if banded.plausibility is None:
    raise ValueError("A condition response must carry its plausibility block")
print("log density of the pinned value:", banded.plausibility.equality_log_density)
print("log probability of the yield band:", banded.plausibility.region_log_probability)
if banded.diagnostics.interval_estimator is not None:
    raise ValueError("A single interval column is exact and reports no estimator")
band = banded.to_pandas_wide()
outside_band = band[
    (band["treasury_10y_yield"] < yield_lower - BAND_TOLERANCE)
    | (band["treasury_10y_yield"] > yield_upper + BAND_TOLERANCE)
]
if not outside_band.empty:
    raise ValueError(f"A banded column left its own interval:\n{outside_band}")
band

## When the region has to be estimated
Bounding the FX rate as well turns the region into a box. That has no closed form, so the service estimates its probability with a quasi-random walk and only then reports the accounting in `diagnostics.interval_estimator`: `points` is how many quasi-random points went into the estimate and `effective_sample_size` is Kish's effective sample size of the weights behind them. A small effective sample size against a large point count means a deep-tail box where few points carry the answer. The cell asserts both sides of that rule — absent for the single band above, present here.

Drawing from a boxed conditional also shows what the box does to the draws. Each row pairs one truncated draw of the bounded columns with a read-out drawn from *that* draw's conditional, so the rows are exact draws from the joint conditional rather than separately summarized margins, and every one of them lands inside both bands.

In [ ]:
FX_BAND_HALF_WIDTH = 0.004

last_fx_rate = float(history["eur_usd_rate"].iloc[-1])
fx_lower = last_fx_rate - FX_BAND_HALF_WIDTH
fx_upper = last_fx_rate + FX_BAND_HALF_WIDTH
fx_band = IntervalCondition(column="eur_usd_rate", lower=fx_lower, upper=fx_upper)
rally_with_two_bands = ConditionBlock(
    query_time_index=CONDITIONED_STEP,
    conditions=[
        EqualityCondition(column=PINNED_COLUMN, value=rallied_equity_level),
        yield_band,
        fx_band,
    ],
)
boxed = client.forecast_samples(
    history,
    query_times=QUERY_TIMES,
    requested_columns=["treasury_10y_yield", "eur_usd_rate", *plan.requested_columns],
    columns=plan.columns,
    n_samples=N_SAMPLES,
    seed=7,
    condition=rally_with_two_bands,
)
if boxed.plausibility is None:
    raise ValueError("A condition response must carry its plausibility block")
estimator = boxed.diagnostics.interval_estimator
if estimator is None:
    raise ValueError(
        "A multi-column region is estimated and must report its accounting"
    )
print(
    "log probability of the box given the pin:",
    boxed.plausibility.region_log_probability,
)
print("estimator points:", estimator.points)
print("effective sample size:", estimator.effective_sample_size)
box_draws = boxed.to_pandas_wide()
outside_box = box_draws[
    (box_draws["treasury_10y_yield"] < yield_lower - BAND_TOLERANCE)
    | (box_draws["treasury_10y_yield"] > yield_upper + BAND_TOLERANCE)
    | (box_draws["eur_usd_rate"] < fx_lower - BAND_TOLERANCE)
    | (box_draws["eur_usd_rate"] > fx_upper + BAND_TOLERANCE)
]
if not outside_box.empty:
    raise ValueError(f"A draw left the box it was conditioned into:\n{outside_box}")
box_draws

## Ranking candidate scenarios
The plausibility section above ended on the rule that only differences of `equality_log_density` are unit-free. This is what that buys: the gap between two pins on the same column is a log likelihood ratio, so candidate what-ifs can be ordered even though no single one of them can be thresholded.

The table asks the model for the conditional forecast under several rallies and reports each pin's plausibility relative to the most plausible one, so a scenario the model finds far-fetched is visible next to the answer it produced. The mean is the right read-out here precisely because the question is not about one scenario's shape: it needs one comparable number per candidate.

In [ ]:
CANDIDATE_RALLIES = [0.94, 0.98, 1.00, 1.02, 1.06]

rankings = []
for candidate in CANDIDATE_RALLIES:
    candidate_level = last_equity_level * candidate
    candidate_result = client.forecast_mean(
        history,
        query_times=QUERY_TIMES,
        requested_columns=plan.requested_columns,
        columns=plan.columns,
        seed=7,
        condition=ConditionBlock(
            query_time_index=CONDITIONED_STEP,
            conditions=[EqualityCondition(column=PINNED_COLUMN, value=candidate_level)],
        ),
    )
    if candidate_result.plausibility is None:
        raise ValueError("A condition response must carry its plausibility block")
    conditional_mean = candidate_result.to_pandas_wide().iloc[0]
    rankings.append(
        {
            "rally": candidate,
            PINNED_COLUMN: candidate_level,
            "equality_log_density": candidate_result.plausibility.equality_log_density,
            **{column: conditional_mean[column] for column in plan.requested_columns},
        }
    )

ranking = pd.DataFrame.from_records(rankings)
ranking["log_ratio_vs_best"] = (
    ranking["equality_log_density"] - ranking["equality_log_density"].max()
)
ranking.sort_values("log_ratio_vs_best", ascending=False, ignore_index=True)

## Scoring an outcome you already have
Every read-out so far answers *what does the model expect*. `log_prob` asks the opposite — how plausible are values I already hold — and is the only return mode where the caller supplies the future instead of receiving it. Under a condition it scores those values against the conditional, which is how two candidate outcomes of the same what-if are compared.

Three rules follow from scoring a joint rather than a projection of one, and the client enforces all three before anything is sent. `query_rows` carries one observed row per entry of `query_times`, each with a value for every declared column. `requested_columns` must name every *readable* column in schema order — a narrower projection would score a different distribution than the caller means to ask about — and the pinned column is the one exception, since conditioning fixed its value. The deployment refuses a row that contradicts the condition, a pinned column given another value or a banded column outside its range, instead of scoring it.

The scores below are log densities of whole rows, so the unit caveat from the plausibility section applies unchanged: read their difference, not their level.

In [ ]:
SCORED_QUERY_TIMES = QUERY_TIMES[: CONDITIONED_STEP + 1]
STRESS_NAV_DROP = 0.9

last_row = history.iloc[-1]
continuation = {
    PINNED_COLUMN: rallied_equity_level,
    "treasury_10y_yield": last_yield,
    "eur_usd_rate": last_fx_rate,
    "portfolio_nav": float(last_row["portfolio_nav"]),
    "realized_volatility": float(last_row["realized_volatility"]),
}
stressed = {
    **continuation,
    "portfolio_nav": continuation["portfolio_nav"] * STRESS_NAV_DROP,
}

scores = {}
for label, outcome in {"continuation": continuation, "stressed": stressed}.items():
    scored = client.forecast_log_prob(
        history,
        query_times=SCORED_QUERY_TIMES,
        query_rows=pd.DataFrame([outcome], columns=EXPECTED_COLUMNS),
        requested_columns=READABLE_COLUMNS,
        columns=plan.columns,
        seed=7,
        condition=rally,
    )
    scores[label] = scored.log_prob.total

print("log density of the continuation:", scores["continuation"])
print("log density of the stressed outcome:", scores["stressed"])
print("log likelihood ratio:", scores["continuation"] - scores["stressed"])